# Sliced Bread: a personal budget tracker

Add your income and spending; the app shows how much is left to invest each month, warns you near and over
your limits, compares your split with targets like 50/30/20, and tracks a savings goal.

**To run it:** Runtime > Run all. The app opens at the bottom of this notebook. A public `gradio.live`
link is printed above it as well.

Every source file is below as its own cell. Each cell starts with `%%writefile`, which saves the cell as
that `.py` file so the files can import each other. This notebook is generated from the files in
[github.com/sourypink/Sliced-Bread](https://github.com/sourypink/Sliced-Bread) by `build_notebook.py`.

In [ ]:
# Install the exact versions the app was tested with.
!pip -q install gradio==6.31.0 pytest==9.2.0

## `budget.py`

The math: totals, what is left to invest, limit and target alerts, input checks. No Gradio here.

In [ ]:
%%writefile budget.py
"""Budget math for Sliced Bread: pure functions, no Gradio, all money in whole cents."""

from datetime import date

# The page reads top to bottom in this order; investments are computed, never stored.
SECTIONS = ("income", "necessities", "discretionary", "travel")
EXPENSE_SECTIONS = ("necessities", "discretionary", "travel")
LIMIT_KEYS = ("overall",) + EXPENSE_SECTIONS

# Percentage targets: which sections count as needs and wants. Savings is whatever is left.
TARGET_KEYS = ("needs", "wants", "savings")
TARGET_GROUPS = {"needs": ("necessities",), "wants": ("discretionary", "travel")}
RULE_50_30_20 = {"needs": 50, "wants": 30, "savings": 20}
PERCENT_WHOLE = 100

WARN_AT = 0.9  # share of a limit at which the meter turns to "near limit"
MAX_AMOUNT_CENTS = 10_000_000 * 100  # $10M; anything larger is a typo
MAX_NAME_LENGTH = 40  # characters
CENTS_PER_DOLLAR = 100


def empty_month():
    """A month with every section present and empty."""
    return {section: [] for section in SECTIONS}


def empty_budget():
    """A new user's budget: no limits, no goal, no months."""
    return {
        "settings": {"savings_goal": 0, "limits": {key: 0 for key in LIMIT_KEYS}, "targets": {}},
        "months": {},
    }


def parse_amount(text):
    """Turn what the user typed into cents, or raise ValueError saying what is wrong."""
    cleaned = str(text if text is not None else "").strip().replace("$", "").replace(",", "")
    if not cleaned:
        raise ValueError("Enter an amount")
    if cleaned.startswith("-"):
        raise ValueError("Amount can't be negative")
    whole, _, fraction = cleaned.partition(".")
    if not (whole or fraction) or not (whole or "0").isdigit() or (fraction and not fraction.isdigit()):
        raise ValueError("Amount must be a number, like 12.50")
    if len(fraction) > 2:
        raise ValueError("Amount can have at most 2 decimal places")
    cents = int(whole or 0) * CENTS_PER_DOLLAR + int(fraction.ljust(2, "0"))
    if cents == 0:
        raise ValueError("Amount must be more than $0")
    if cents > MAX_AMOUNT_CENTS:
        raise ValueError("Amount is too large")
    return cents


def parse_optional_amount(text):
    """Like parse_amount, but blank or 0 means "not set" and returns 0."""
    cleaned = str(text if text is not None else "").strip().replace("$", "").replace(",", "")
    if cleaned.strip("0.") == "":
        return 0
    return parse_amount(cleaned)


def clean_name(text):
    """Trim a line item's name, or raise ValueError if it is unusable."""
    name = " ".join(str(text if text is not None else "").split())
    if not name:
        raise ValueError("Enter a name")
    if len(name) > MAX_NAME_LENGTH:
        raise ValueError(f"Name must be {MAX_NAME_LENGTH} characters or fewer")
    return name


def format_money(cents):
    """$1,200 for whole dollars, $12.50 otherwise, -$85 when negative."""
    sign = "-" if cents < 0 else ""
    dollars, remainder = divmod(abs(cents), CENTS_PER_DOLLAR)
    if remainder:
        return f"{sign}${dollars:,}.{remainder:02d}"
    return f"{sign}${dollars:,}"


def section_total(month, section):
    """Sum of every line in one section."""
    return sum(item["amount"] for item in month.get(section, []))


def total_expenses(month):
    """Necessities + discretionary + travel."""
    return sum(section_total(month, section) for section in EXPENSE_SECTIONS)


def leftover(month):
    """Income minus expenses: what goes to investments. Negative means over budget."""
    return section_total(month, "income") - total_expenses(month)


def meter(spent, limit):
    """State of a spending limit: none (no limit set), good, warn or over, with its label."""
    if limit <= 0:
        return {"state": "none", "fraction": 0.0, "label": ""}
    fraction = spent / limit
    if spent > limit:
        return {"state": "over", "fraction": 1.0, "label": f"Over by {format_money(spent - limit)}"}
    if fraction >= WARN_AT:
        return {"state": "warn", "fraction": fraction, "label": f"Near limit, {format_money(limit - spent)} left"}
    return {"state": "good", "fraction": fraction, "label": f"{format_money(limit - spent)} left"}


def savings_progress(budget):
    """Saved so far: the positive leftovers of every month added up."""
    return sum(max(0, leftover(month)) for month in budget["months"].values())


def add_item(budget, month_key, section, name_text, amount_text):
    """Validate and add a line to a section; returns the added item."""
    if section not in SECTIONS:
        raise ValueError(f"Unknown section: {section}")
    item = {"name": clean_name(name_text), "amount": parse_amount(amount_text)}
    month = budget["months"].setdefault(month_key, empty_month())
    month[section].append(item)
    return item


def remove_item(budget, month_key, section, index):
    """Delete one line from a section by its position."""
    items = budget["months"].get(month_key, empty_month())[section]
    if not 0 <= index < len(items):
        raise IndexError(f"No line {index} in {section}")
    items.pop(index)


def update_item(budget, month_key, section, index, name_text, amount_text):
    """Validate and replace one line; returns the new item."""
    items = budget["months"].get(month_key, empty_month())[section]
    if not 0 <= index < len(items):
        raise IndexError(f"No line {index} in {section}")
    item = {"name": clean_name(name_text), "amount": parse_amount(amount_text)}
    items[index] = item
    return item


def parse_targets(needs_text, wants_text, savings_text):
    """Turn three typed percentages into targets. All blank means targets are off ({})."""
    typed = dict(zip(TARGET_KEYS, (needs_text, wants_text, savings_text)))
    cleaned = {key: str(text if text is not None else "").strip().rstrip("%").strip() for key, text in typed.items()}
    if not any(cleaned.values()):
        return {}
    targets = {}
    for key, text in cleaned.items():
        if not text.isdigit() or int(text) > PERCENT_WHOLE:
            raise ValueError(f"{key.capitalize()} must be a whole number from 0 to 100")
        targets[key] = int(text)
    total = sum(targets.values())
    if total != PERCENT_WHOLE:
        raise ValueError(f"Targets must add up to 100% (now {total}%)")
    return targets


def target_meters(targets, month):
    """Each target's meter, with the actual and target percent of income. Empty when off or no income."""
    income = section_total(month, "income")
    if not targets or income <= 0:
        return {}
    meters = {}
    for key, sections in TARGET_GROUPS.items():
        spent = sum(section_total(month, section) for section in sections)
        result = meter(spent, income * targets[key] // PERCENT_WHOLE)
        meters[key] = {**result, "actual_percent": round(spent * PERCENT_WHOLE / income), "target_percent": targets[key]}
    saved = max(leftover(month), 0)
    goal = income * targets["savings"] // PERCENT_WHOLE
    if goal <= 0:
        result = {"state": "none", "fraction": 0.0, "label": ""}
    elif saved >= goal:
        result = {"state": "good", "fraction": 1.0, "label": "On target"}
    else:
        result = {"state": "warn", "fraction": saved / goal, "label": f"{format_money(goal - saved)} short"}
    meters["savings"] = {**result, "actual_percent": round(saved * PERCENT_WHOLE / income), "target_percent": targets["savings"]}
    return meters


def get_month(budget, month_key):
    """The month's data, or an empty month if nothing was entered yet."""
    return budget["months"].get(month_key, empty_month())


def month_key(day=None):
    """'2026-10' for any date in October 2026; today by default."""
    day = day or date.today()
    return f"{day.year}-{day.month:02d}"


def shift_month(key, steps):
    """Move a month key forward or back: shift_month('2026-12', 1) is '2027-01'."""
    year, month = (int(part) for part in key.split("-"))
    index = year * 12 + (month - 1) + steps
    return f"{index // 12}-{index % 12 + 1:02d}"


def month_name(key):
    """'October' for '2026-10'."""
    year, month = (int(part) for part in key.split("-"))
    return date(year, month, 1).strftime("%B")


def month_label(key):
    """'October 2026' for '2026-10'."""
    return f"{month_name(key)} {key.split('-')[0]}"


## `storage.py`

Accounts (salted, hashed passwords) and saving each user's budget as a JSON file.

In [ ]:
%%writefile storage.py
"""Accounts and saved budgets for Sliced Bread, kept as JSON files in DATA_DIR."""

import hashlib
import json
import os
import re
import secrets
from datetime import date

import budget

# Set SLICED_BREAD_DATA to a Google Drive folder in Colab so data survives a runtime reset.
DATA_DIR = os.environ.get("SLICED_BREAD_DATA", "data")
USERS_FILE = "users.json"
HASH_ROUNDS = 200_000  # PBKDF2 iterations; slow enough to resist guessing, fast enough to sign in
MIN_PASSWORD_LENGTH = 6  # characters
USERNAME_PATTERN = re.compile(r"[a-z0-9_]{3,20}")  # also safe as a file name


def _path(file_name):
    return os.path.join(DATA_DIR, file_name)


def _read_json(file_name, default):
    try:
        with open(_path(file_name), encoding="utf-8") as file:
            return json.load(file)
    except FileNotFoundError:
        return default


def _write_json(file_name, content):
    """Write to a temp file first, then swap it in, so a crash never leaves half a file."""
    os.makedirs(DATA_DIR, exist_ok=True)
    temp_path = _path(file_name + ".tmp")
    with open(temp_path, "w", encoding="utf-8") as file:
        json.dump(content, file, indent=2)
    os.replace(temp_path, _path(file_name))


def _hash_password(password, salt):
    return hashlib.pbkdf2_hmac("sha256", password.encode(), bytes.fromhex(salt), HASH_ROUNDS).hex()


def _check_password_rules(password):
    if len(password or "") < MIN_PASSWORD_LENGTH:
        raise ValueError(f"Password must be at least {MIN_PASSWORD_LENGTH} characters")


def normalize_username(text):
    """Lowercase and trim a username, or raise ValueError if it breaks the rules."""
    username = (text or "").strip().lower()
    if not USERNAME_PATTERN.fullmatch(username):
        raise ValueError("Username must be 3-20 letters, numbers or underscores")
    return username


def create_account(username_text, password, display_name=""):
    """Create a user and an empty budget; returns the username."""
    username = normalize_username(username_text)
    _check_password_rules(password)
    users = _read_json(USERS_FILE, {})
    if username in users:
        raise ValueError("That username is taken")
    salt = secrets.token_hex(16)
    users[username] = {
        "display_name": (display_name or "").strip() or username,
        "salt": salt,
        "password_hash": _hash_password(password, salt),
        "created": date.today().isoformat(),
    }
    _write_json(USERS_FILE, users)
    save_budget(username, budget.empty_budget())
    return username


def sign_in(username_text, password):
    """Return the username if the password matches, otherwise raise ValueError."""
    users = _read_json(USERS_FILE, {})
    username = (username_text or "").strip().lower()
    user = users.get(username)
    # Same message for both failures, so the screen doesn't reveal which usernames exist.
    if not user or not secrets.compare_digest(_hash_password(password or "", user["salt"]), user["password_hash"]):
        raise ValueError("Wrong username or password")
    return username


def change_password(username, current_password, new_password):
    """Replace a password after checking the current one."""
    try:
        sign_in(username, current_password)
    except ValueError:
        raise ValueError("Wrong current password") from None
    _check_password_rules(new_password)
    users = _read_json(USERS_FILE, {})
    salt = secrets.token_hex(16)
    users[username].update(salt=salt, password_hash=_hash_password(new_password, salt))
    _write_json(USERS_FILE, users)


def get_display_name(username):
    return _read_json(USERS_FILE, {})[username]["display_name"]


def set_display_name(username, display_name):
    users = _read_json(USERS_FILE, {})
    users[username]["display_name"] = (display_name or "").strip() or username
    _write_json(USERS_FILE, users)


def load_budget(username):
    return _read_json(f"{username}.json", budget.empty_budget())


def save_budget(username, user_budget):
    _write_json(f"{username}.json", user_budget)


def delete_account(username):
    """Remove the user and their budget file for good."""
    users = _read_json(USERS_FILE, {})
    users.pop(username, None)
    _write_json(USERS_FILE, users)
    budget_path = _path(f"{username}.json")
    if os.path.exists(budget_path):
        os.remove(budget_path)


## `design.py`

Toast, the design system: colours, type sizes, spacing, the Gradio theme and the CSS.

In [ ]:
%%writefile design.py
"""Toast, the Sliced Bread design system: tokens, the Gradio theme and the CSS. See PLAN.md section 5.

The one rule: crust means "you can press it". Crust marks controls only, never money or status.
Nothing in app.py sets a colour or size by hand; it uses the classes defined here.
"""

import html

import gradio as gr

import budget

# ---------- Tokens ----------

PAPER = "#FAF8F4"  # page background
CARD = "#FFFFFF"  # section cards
INK = "#1F1D1A"  # all text, all money
MUTED = "#6B655C"  # secondary labels, 5.4:1 on paper
LINE = "#E7E2D9"  # borders, dividers, empty meter track
CRUST = "#9A5B1E"  # controls and focus ring; white text on it is 5.4:1
CRUST_DARK = "#7E4916"  # control hover
CRUST_SOFT = "#F6EBDD"  # wash behind the open add row
GOOD = "#2F7D4F"  # meter under 90%, savings bar
WARN = "#D99A00"  # meter 90-100% (fill only, never text)
OVER = "#B42318"  # meter over 100%, "Over budget" text

# Fills only (section dots, the stacked bar), never text.
SECTION_COLORS = {
    "income": INK,
    "necessities": "#3B6FB5",
    "discretionary": "#8E4FA8",
    "travel": "#1E8C93",
    "investments": GOOD,
}
METER_COLORS = {"good": GOOD, "warn": WARN, "over": OVER}

FONT_TEXT = "Inter"
FONT_MONEY = "JetBrains Mono"
PAGE_MAX_WIDTH = "640px"

# Crust and paper as Gradio colour ramps, so built-in components pick them up.
CRUST_RAMP = gr.themes.Color(
    c50="#FBF4EA", c100=CRUST_SOFT, c200="#EBD3B5", c300="#DDB58A", c400="#C88F58", c500="#B07235",
    c600=CRUST, c700=CRUST_DARK, c800="#633912", c900="#4A2B0E", c950="#2E1B09", name="crust",
)
PAPER_RAMP = gr.themes.Color(
    c50=PAPER, c100="#F3F0EA", c200=LINE, c300="#D3CCC0", c400="#A39C90", c500="#7D776D",
    c600=MUTED, c700="#4A4640", c800="#2F2C28", c900=INK, c950="#141311", name="paper",
)


def build_theme():
    """The Gradio theme. Light only: every dark-mode value is copied from its light twin."""
    theme = gr.themes.Base(
        primary_hue=CRUST_RAMP,
        secondary_hue=CRUST_RAMP,
        neutral_hue=PAPER_RAMP,
        font=[gr.themes.GoogleFont(FONT_TEXT), "ui-sans-serif", "system-ui", "sans-serif"],
        font_mono=[gr.themes.GoogleFont(FONT_MONEY), "ui-monospace", "Consolas", "monospace"],
        radius_size=gr.themes.sizes.radius_md,
        text_size=gr.themes.sizes.text_md,
    ).set(
        body_background_fill=PAPER,
        body_text_color=INK,
        body_text_color_subdued=MUTED,
        background_fill_primary=CARD,
        background_fill_secondary=PAPER,
        border_color_primary=LINE,
        block_background_fill=CARD,
        block_border_color=LINE,
        block_shadow="none",
        input_background_fill=CARD,
        input_border_color=LINE,
        input_border_color_focus=CRUST,
        button_primary_background_fill=CRUST,
        button_primary_background_fill_hover=CRUST_DARK,
        button_primary_text_color="#FFFFFF",
        button_primary_border_color=CRUST,
        button_secondary_background_fill=CARD,
        button_secondary_background_fill_hover=CRUST_SOFT,
        button_secondary_text_color=INK,
        button_secondary_border_color=LINE,
        button_cancel_background_fill=OVER,
        button_cancel_background_fill_hover="#8F1C13",
        button_cancel_text_color="#FFFFFF",
    )
    for name in list(vars(theme)):
        if name.endswith("_dark"):
            setattr(theme, name, getattr(theme, name.removesuffix("_dark")))
    return theme


CSS = f"""
:root {{
  --sb-paper: {PAPER}; --sb-card: {CARD}; --sb-ink: {INK}; --sb-muted: {MUTED}; --sb-line: {LINE};
  --sb-crust: {CRUST}; --sb-crust-dark: {CRUST_DARK}; --sb-crust-soft: {CRUST_SOFT};
  --sb-over: {OVER};
  --sb-money: '{FONT_MONEY}', ui-monospace, Consolas, monospace;
}}

/* One centred column, phone first. */
.gradio-container {{ width: 100% !important; max-width: {PAGE_MAX_WIDTH} !important; margin: 0 auto !important; padding: 16px !important; }}
.gradio-container .main {{ padding: 0 !important; }}
footer {{ display: none !important; }}
:focus-visible {{ outline: 3px solid var(--sb-crust) !important; outline-offset: 2px; }}

/* Type scale: 14 / 16 / 20 / 24 / 32 / 40. Nothing smaller than 14. The hero is 40, or 32 on a phone. */
.sb-brand {{ font-size: 24px; font-weight: 700; letter-spacing: -0.01em; }}
.sb-subtitle {{ font-size: 16px; color: var(--sb-muted); margin-top: 4px; }}
.sb-hero {{ font-size: 40px; line-height: 1.1; font-weight: 700; letter-spacing: -0.02em; text-wrap: balance; }}
.sb-hero.sb-over {{ color: var(--sb-over); }}
@media (max-width: 480px) {{ .sb-hero {{ font-size: 32px; }} }}
.sb-title {{ font-size: 20px; font-weight: 600; display: flex; align-items: center; gap: 8px; }}
.sb-note {{ font-size: 14px; color: var(--sb-muted); }}
.sb-money {{ font-family: var(--sb-money); font-variant-numeric: tabular-nums; white-space: nowrap; }}
.sb-dot {{ width: 10px; height: 10px; border-radius: 999px; display: inline-block; flex: none; }}

/* Cards: 12px radius, 16px padding, one shadow. */
.sb-card {{
  background: var(--sb-card); border: 1px solid var(--sb-line); border-radius: 12px;
  padding: 16px !important; gap: 8px !important; box-shadow: 0 1px 2px rgb(0 0 0 / 0.06);
}}
.sb-card .html-container, .sb-card .prose {{ padding: 0 !important; }}
.sb-row {{ align-items: center !important; gap: 8px !important; flex-wrap: nowrap !important; }}
.sb-grow {{ flex: 1 1 auto !important; min-width: 0 !important; }}
.sb-head {{ display: flex; justify-content: space-between; align-items: baseline; gap: 12px; }}
.sb-head .sb-money {{ font-size: 16px; }}
.sb-line {{
  display: flex; justify-content: space-between; gap: 12px; font-size: 16px;
  padding: 8px 0; border-top: 1px solid var(--sb-line);
}}
.sb-line-name {{ overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }}

/* Meters: colour in the fill, words beside it. */
.sb-meter {{ height: 8px; background: var(--sb-line); border-radius: 999px; overflow: hidden; }}
.sb-meter > span {{ display: block; height: 100%; border-radius: 999px; }}
.sb-meter-row {{ display: grid; gap: 4px; }}
.sb-split {{ display: grid; gap: 12px; margin-top: 12px; }}
.sb-meter-label {{ font-size: 14px; color: var(--sb-ink); display: flex; justify-content: space-between; gap: 8px; }}

/* Where the money went: one stacked bar and its key. */
.sb-bar {{ display: flex; height: 16px; border-radius: 999px; overflow: hidden; background: var(--sb-line); margin: 16px 0 8px; }}
.sb-key {{ display: flex; flex-wrap: wrap; gap: 4px 16px; font-size: 14px; }}
.sb-key span {{ display: inline-flex; align-items: center; gap: 6px; }}

/* Pill controls: + and x. Crust on the +, quiet on the x. */
.sb-plus {{ border-radius: 999px !important; min-width: 40px !important; max-width: 40px; height: 40px; padding: 0 !important; font-size: 20px !important; }}
.sb-x {{
  border-radius: 999px !important; min-width: 32px !important; max-width: 32px; height: 32px; padding: 0 !important;
  border: none !important; background: transparent !important; color: var(--sb-muted) !important; font-size: 16px !important;
}}
.sb-x:hover {{ background: var(--sb-crust-soft) !important; color: var(--sb-ink) !important; }}
.sb-add {{ background: var(--sb-crust-soft); border-radius: 8px; padding: 12px !important; gap: 8px !important; }}
.sb-month {{ align-items: center !important; justify-content: space-between; flex-wrap: nowrap !important; }}
.sb-month .sb-title {{ justify-content: center; }}
.sb-gap {{ gap: 12px !important; }}
"""


# ---------- HTML pieces (escape everything the user typed) ----------

def dot(section):
    return f'<span class="sb-dot" style="background:{SECTION_COLORS[section]}"></span>'


def money(cents):
    return f'<span class="sb-money">{budget.format_money(cents)}</span>'


def meter_html(result, label_left=""):
    """A limit meter with its words. Empty string when no limit is set."""
    if result["state"] == "none":
        return ""
    width = max(result["fraction"] * 100, 2)
    return (
        f'<div class="sb-meter-row"><div class="sb-meter"><span style="width:{width:.1f}%;'
        f'background:{METER_COLORS[result["state"]]}"></span></div>'
        f'<div class="sb-meter-label"><span>{html.escape(label_left)}</span><span>{html.escape(result["label"])}</span></div></div>'
    )


def progress_html(saved, goal):
    """The savings goal bar: always the good colour, it only fills."""
    width = max(min(saved / goal, 1) * 100, 2)
    return (
        f'<div class="sb-meter-row"><div class="sb-meter"><span style="width:{width:.1f}%;background:{GOOD}"></span></div>'
        f'<div class="sb-meter-label"><span>Savings goal</span><span class="sb-money">'
        f'{budget.format_money(saved)} of {budget.format_money(goal)}</span></div></div>'
    )


def line_html(name, cents):
    return f'<div class="sb-line"><span class="sb-line-name">{html.escape(name)}</span>{money(cents)}</div>'


## `app.py`

The Gradio screens: sign in, the Budget tab and the Account tab. The only file that uses Gradio.

In [ ]:
%%writefile app.py
"""Sliced Bread: a personal budget tracker. Run with: python app.py"""

import copy
import sys
from html import escape

import gradio as gr

import budget
import design
import storage

SECTION_NAMES = {
    "income": "Income",
    "necessities": "Necessities",
    "discretionary": "Discretionary",
    "travel": "Travel",
    "investments": "Investments",
}
EMPTY_STATES = {
    "income": "No income yet. Add your salary, freelance or other pay.",
    "necessities": "Nothing yet. Add rent, groceries, gas.",
    "discretionary": "Nothing yet. Add movies, events, games, subscriptions.",
    "travel": "No trips yet. Add flights, hotels, food.",
}
NAME_PLACEHOLDERS = {"income": "Salary", "necessities": "Rent", "discretionary": "Concert tickets", "travel": "Flights"}
LIMIT_LABELS = {"overall": "Monthly limit", **SECTION_NAMES}
TARGET_LABELS = {"needs": "Needs", "wants": "Wants", "savings": "Savings"}

def html(value, **options):
    return gr.HTML(value, padding=False, **options)


def fail(message):
    """Show a validation problem to the user without a traceback in the console."""
    return gr.Error(message, print_exception=False)


# ---------- Page pieces ----------

def header_html(display_name):
    return f'<div class="sb-head"><div class="sb-brand">Sliced Bread</div><div class="sb-note">Hi, {escape(display_name)}</div></div>'


def hero_html(user_budget, key):
    """The one hero line plus where the money went."""
    month = budget.get_month(user_budget, key)
    income = budget.section_total(month, "income")
    expenses = budget.total_expenses(month)
    left = budget.leftover(month)
    name = budget.month_name(key)

    if income == 0 and expenses == 0:
        return f'<div class="sb-hero">Add your income to plan {name}</div>'
    if left >= 0:
        title = f'<div class="sb-hero">{design.money(left)} left to invest in {name}</div>'
    else:
        title = f'<div class="sb-hero sb-over">Over budget by {design.money(-left)} in {name}</div>'

    amounts = {section: budget.section_total(month, section) for section in budget.EXPENSE_SECTIONS}
    amounts["investments"] = max(left, 0)
    whole = max(income, expenses)
    segments = "".join(
        f'<span style="width:{cents / whole * 100:.2f}%;background:{design.SECTION_COLORS[section]}"></span>'
        for section, cents in amounts.items() if cents
    )
    key_items = "".join(
        f"<span>{design.dot(section)}{SECTION_NAMES[section]} {design.money(cents)}</span>"
        for section, cents in amounts.items() if cents
    )
    limit = user_budget["settings"]["limits"]["overall"]
    limit_meter = design.meter_html(
        budget.meter(expenses, limit), f"Spent {budget.format_money(expenses)} of {budget.format_money(limit)}"
    )
    return f'{title}<div class="sb-bar">{segments}</div><div class="sb-key">{key_items}</div>' + (
        f'<div style="margin-top:16px">{limit_meter}</div>' if limit_meter else ""
    )


def section_head_html(user_budget, month, section):
    total = budget.section_total(month, section)
    limit = user_budget["settings"]["limits"].get(section, 0)
    amount = design.money(total)
    if limit:
        amount = f'<span class="sb-money">{budget.format_money(total)} of {budget.format_money(limit)}</span>'
    return f'<div class="sb-head"><div class="sb-title">{design.dot(section)}{SECTION_NAMES[section]}</div>{amount}</div>'


def investments_html(user_budget, key):
    left = budget.leftover(budget.get_month(user_budget, key))
    head = (
        f'<div class="sb-head"><div class="sb-title">{design.dot("investments")}Investments</div>'
        f"{design.money(max(left, 0))}</div>"
    )
    if left < 0:
        note = f'<div class="sb-note" style="color:var(--sb-over)">Over budget by {budget.format_money(-left)}. Nothing left to invest.</div>'
    else:
        note = '<div class="sb-note">Whatever is left after spending goes here.</div>'
    goal = user_budget["settings"]["savings_goal"]
    if goal:
        savings = design.progress_html(budget.savings_progress(user_budget), goal)
    else:
        savings = '<div class="sb-note">Set a savings goal on the Account tab.</div>'
    return f'{head}{note}<div style="margin-top:8px">{savings}</div>'


def split_html(user_budget, month):
    """Needs, wants and savings against the user's percentage targets. Empty when targets are off."""
    targets = user_budget["settings"].get("targets", {})
    meters = budget.target_meters(targets, month)
    if not meters:
        return ""
    plan = "/".join(str(targets[key]) for key in budget.TARGET_KEYS)
    rows = "".join(
        design.meter_html(result, f"{TARGET_LABELS[key]} {result['actual_percent']}% of {result['target_percent']}%")
        for key, result in meters.items()
    )
    return f'<div class="sb-title">Your {plan} split</div><div class="sb-split">{rows}</div>'


def watched_meters(user_budget, key):
    """Every limit and target an alert watches, by the name the alert shows."""
    month = budget.get_month(user_budget, key)
    limits = user_budget["settings"]["limits"]
    spent = {"overall": budget.total_expenses(month)}
    spent.update({section: budget.section_total(month, section) for section in budget.EXPENSE_SECTIONS})
    meters = {LIMIT_LABELS[limit_key]: budget.meter(spent[limit_key], limits[limit_key]) for limit_key in budget.LIMIT_KEYS}
    targets = budget.target_meters(user_budget["settings"].get("targets", {}), month)
    meters.update({f"{TARGET_LABELS[target]} target": result for target, result in targets.items() if target != "savings"})
    return meters


def budget_alerts(before, after, key):
    """Warn when a change pushes a limit or target into "near" or "over"."""
    old = watched_meters(before, key)
    for label, new in watched_meters(after, key).items():
        if new["state"] in ("warn", "over") and new["state"] != old.get(label, {"state": "none"})["state"]:
            gr.Warning(f"{label}: {new['label']}", title="Budget alert")


# ---------- Event handlers ----------

def make_toggle(section):
    def toggle(open_section):
        return None if open_section == ("add", section) else ("add", section)
    return toggle


def make_open(value):
    return lambda: value


def make_add(section):
    def add(user, user_budget, key, name, amount):
        updated = copy.deepcopy(user_budget)
        try:
            budget.add_item(updated, key, section, name, amount)
        except ValueError as error:
            raise fail(str(error))
        storage.save_budget(user, updated)
        budget_alerts(user_budget, updated, key)
        return updated, "", ""
    return add


def make_update(section, index):
    def update(user, user_budget, key, name, amount):
        updated = copy.deepcopy(user_budget)
        try:
            budget.update_item(updated, key, section, index, name, amount)
        except ValueError as error:
            raise fail(str(error))
        storage.save_budget(user, updated)
        budget_alerts(user_budget, updated, key)
        return updated, None
    return update


def make_remove(section, index):
    def remove(user, user_budget, key, open_section):
        updated = copy.deepcopy(user_budget)
        budget.remove_item(updated, key, section, index)
        storage.save_budget(user, updated)
        # An open edit row points at a line position that just moved, so close it; an add row can stay.
        return updated, open_section if open_section and open_section[0] == "add" else None
    return remove


def dollars_text(cents):
    """Pre-fill for a limit box: "1,800", or blank when not set."""
    return budget.format_money(cents).removeprefix("$") if cents else ""


def signed_in(username):
    """Everything the screen needs once someone is in: states, views and account fields."""
    user_budget = storage.load_budget(username)
    settings = user_budget["settings"]
    display_name = storage.get_display_name(username)
    return (
        username, user_budget, budget.month_key(), None,
        gr.update(visible=False), gr.update(visible=True),
        header_html(display_name),
        display_name,
        *[dollars_text(settings["limits"][key]) for key in budget.LIMIT_KEYS],
        dollars_text(settings["savings_goal"]),
        *target_texts(settings.get("targets", {})),
        "", "",
    )


def target_texts(targets):
    """Pre-fill for the three target boxes: "50", "30", "20", or blanks when off."""
    return [str(targets[key]) if targets else "" for key in budget.TARGET_KEYS]


def sign_in(username, password):
    try:
        return signed_in(storage.sign_in(username, password))
    except ValueError as error:
        raise fail(str(error))


def create_account(display_name, username, password):
    try:
        return signed_in(storage.create_account(username, password, display_name))
    except ValueError as error:
        raise fail(str(error))


def sign_out():
    return None, None, budget.month_key(), None, gr.update(visible=True), gr.update(visible=False), "", ""


def save_settings(user, user_budget, overall, necessities, discretionary, travel, goal):
    updated = copy.deepcopy(user_budget)
    typed = dict(zip(budget.LIMIT_KEYS, (overall, necessities, discretionary, travel)))
    try:
        for key, text in typed.items():
            updated["settings"]["limits"][key] = parse_field(LIMIT_LABELS[key], text)
        updated["settings"]["savings_goal"] = parse_field("Savings goal", goal)
    except ValueError as error:
        raise fail(str(error))
    storage.save_budget(user, updated)
    gr.Info("Limits and goal saved", title="Saved")
    return updated


def save_targets(user, user_budget, needs, wants, savings):
    try:
        targets = budget.parse_targets(needs, wants, savings)
    except ValueError as error:
        raise fail(str(error))
    updated = copy.deepcopy(user_budget)
    updated["settings"]["targets"] = targets
    storage.save_budget(user, updated)
    gr.Info("Targets saved" if targets else "Targets turned off", title="Saved")
    return updated, *target_texts(targets)


def use_50_30_20(user, user_budget):
    return save_targets(user, user_budget, *(str(percent) for percent in budget.RULE_50_30_20.values()))


def parse_field(label, text):
    try:
        return budget.parse_optional_amount(text)
    except ValueError as error:
        raise ValueError(f"{label}: {error}")


def save_display_name(user, display_name):
    storage.set_display_name(user, display_name)
    gr.Info("Name saved", title="Saved")
    name = storage.get_display_name(user)
    return header_html(name), name


def change_password(user, current, new):
    try:
        storage.change_password(user, current, new)
    except ValueError as error:
        raise fail(str(error))
    gr.Info("Password changed", title="Saved")
    return "", ""


def delete_account(user):
    storage.delete_account(user)
    gr.Info("Account and data deleted", title="Deleted")
    return (*sign_out(), gr.update(visible=False))


# ---------- Layout ----------

with gr.Blocks(title="Sliced Bread") as demo:
    user_state = gr.State(None)  # signed-in username
    budget_state = gr.State(None)  # that user's whole budget
    month_state = gr.State(budget.month_key())  # month on screen, "2026-10"
    open_state = gr.State(None)  # ("add", section) or ("edit", section, index): the one open form

    # Sign in / create account
    with gr.Column(elem_classes="sb-card") as auth_view:
        html('<div class="sb-brand">Sliced Bread</div><div class="sb-subtitle">Know what is left to invest each month</div>')
        with gr.Tabs():
            with gr.Tab("Sign in"):
                login_username = gr.Textbox(label="Username", html_attributes=gr.InputHTMLAttributes(autocomplete="username", autocapitalize="off"))
                login_password = gr.Textbox(label="Password", type="password", html_attributes=gr.InputHTMLAttributes(autocomplete="current-password"))
                login_button = gr.Button("Sign in", variant="primary")
            with gr.Tab("Create account"):
                new_display_name = gr.Textbox(label="Your name")
                new_username = gr.Textbox(label="Username", placeholder="3-20 letters, numbers or _", html_attributes=gr.InputHTMLAttributes(autocomplete="username", autocapitalize="off"))
                new_password = gr.Textbox(label="Password", type="password", placeholder="At least 6 characters", html_attributes=gr.InputHTMLAttributes(autocomplete="new-password"))
                create_button = gr.Button("Create account", variant="primary")

    with gr.Column(visible=False) as app_view:
        greeting = html("")

        with gr.Tabs():
            with gr.Tab("Budget"):
                with gr.Row(elem_classes="sb-month"):
                    previous_month = gr.Button("‹", elem_classes="sb-x", min_width=0, scale=0)
                    month_title = html("", elem_classes="sb-grow")
                    next_month = gr.Button("›", elem_classes="sb-x", min_width=0, scale=0)

                @gr.render(inputs=[budget_state, month_state, open_state], triggers=[budget_state.change, month_state.change, open_state.change])
                def render_budget(user_budget, key, open_section):
                    if not user_budget:
                        return
                    month = budget.get_month(user_budget, key)
                    opened = tuple(open_section) if open_section else None
                    with gr.Column(elem_classes="sb-card"):
                        html(hero_html(user_budget, key))
                    split = split_html(user_budget, month)
                    if split:
                        with gr.Column(elem_classes="sb-card"):
                            html(split)

                    for section in budget.SECTIONS:
                        is_open = opened == ("add", section)
                        with gr.Column(elem_classes="sb-card"):
                            with gr.Row(elem_classes="sb-row"):
                                html(section_head_html(user_budget, month, section), elem_classes="sb-grow")
                                plus = gr.Button("−" if is_open else "+", variant="primary", elem_classes="sb-plus", min_width=0, scale=0)
                                plus.click(make_toggle(section), inputs=open_state, outputs=open_state)

                            limit = user_budget["settings"]["limits"].get(section, 0)
                            meter = design.meter_html(budget.meter(budget.section_total(month, section), limit))
                            if meter:
                                html(meter)

                            if is_open:
                                with gr.Column(elem_classes="sb-add"):
                                    with gr.Row(elem_classes="sb-row"):
                                        name_box = gr.Textbox(label="Name", placeholder=NAME_PLACEHOLDERS[section], autofocus=True, scale=3, min_width=0)
                                        amount_box = gr.Textbox(label="Amount", placeholder="0.00", scale=2, min_width=0)
                                    with gr.Row(elem_classes="sb-row"):
                                        add_button = gr.Button("Add", variant="primary")
                                        cancel_button = gr.Button("Done")
                                add_inputs = [user_state, budget_state, month_state, name_box, amount_box]
                                for trigger in (add_button.click, amount_box.submit, name_box.submit):
                                    trigger(make_add(section), inputs=add_inputs, outputs=[budget_state, name_box, amount_box])
                                cancel_button.click(lambda: None, outputs=open_state)

                            items = month[section]
                            if not items and not is_open:
                                html(f'<div class="sb-note">{EMPTY_STATES[section]}</div>')
                            for index, item in enumerate(items):
                                if opened == ("edit", section, index):
                                    with gr.Column(elem_classes="sb-add"):
                                        with gr.Row(elem_classes="sb-row"):
                                            edit_name = gr.Textbox(label="Name", value=item["name"], autofocus=True, scale=3, min_width=0)
                                            edit_amount = gr.Textbox(label="Amount", value=dollars_text(item["amount"]), scale=2, min_width=0)
                                        with gr.Row(elem_classes="sb-row"):
                                            save_button = gr.Button("Save", variant="primary")
                                            cancel_edit_button = gr.Button("Cancel")
                                    edit_inputs = [user_state, budget_state, month_state, edit_name, edit_amount]
                                    for trigger in (save_button.click, edit_name.submit, edit_amount.submit):
                                        trigger(make_update(section, index), inputs=edit_inputs, outputs=[budget_state, open_state])
                                    cancel_edit_button.click(lambda: None, outputs=open_state)
                                    continue
                                with gr.Row(elem_classes="sb-row sb-item"):
                                    html(design.line_html(item["name"], item["amount"]), elem_classes="sb-grow")
                                    edit = gr.Button("✎", elem_classes="sb-x", min_width=0, scale=0)
                                    edit.click(make_open(("edit", section, index)), outputs=open_state)
                                    remove = gr.Button("✕", elem_classes="sb-x", min_width=0, scale=0)
                                    remove.click(make_remove(section, index), inputs=[user_state, budget_state, month_state, open_state], outputs=[budget_state, open_state])

                    with gr.Column(elem_classes="sb-card"):
                        html(investments_html(user_budget, key))

            with gr.Tab("Account"):
                with gr.Column(elem_classes="sb-card"):
                    html('<div class="sb-title">Monthly limits</div><div class="sb-note">Leave blank for no limit.</div>')
                    with gr.Row():
                        limit_overall = gr.Textbox(label="Monthly limit", placeholder="No limit", min_width=120)
                        limit_necessities = gr.Textbox(label="Necessities", placeholder="No limit", min_width=120)
                    with gr.Row():
                        limit_discretionary = gr.Textbox(label="Discretionary", placeholder="No limit", min_width=120)
                        limit_travel = gr.Textbox(label="Travel", placeholder="No limit", min_width=120)
                    savings_goal = gr.Textbox(label="Savings goal", placeholder="No goal")
                    save_limits_button = gr.Button("Save limits and goal", variant="primary")

                with gr.Column(elem_classes="sb-card"):
                    html(
                        '<div class="sb-title">Percentage targets</div><div class="sb-note">Needs are necessities, '
                        "wants are discretionary and travel, savings is what is left. Leave blank to turn off.</div>"
                    )
                    with gr.Row():
                        target_needs = gr.Textbox(label="Needs %", placeholder="Off", min_width=80)
                        target_wants = gr.Textbox(label="Wants %", placeholder="Off", min_width=80)
                        target_savings = gr.Textbox(label="Savings %", placeholder="Off", min_width=80)
                    with gr.Row():
                        use_rule_button = gr.Button("Use 50/30/20")
                        save_targets_button = gr.Button("Save targets", variant="primary")

                with gr.Column(elem_classes="sb-card"):
                    html('<div class="sb-title">Profile</div>')
                    display_name = gr.Textbox(label="Your name")
                    save_name_button = gr.Button("Save name")

                with gr.Column(elem_classes="sb-card"):
                    html('<div class="sb-title">Password</div>')
                    current_password = gr.Textbox(label="Current password", type="password", html_attributes=gr.InputHTMLAttributes(autocomplete="current-password"))
                    changed_password = gr.Textbox(label="New password", type="password", html_attributes=gr.InputHTMLAttributes(autocomplete="new-password"))
                    change_password_button = gr.Button("Change password")

                with gr.Column(elem_classes="sb-card"):
                    sign_out_button = gr.Button("Sign out")
                    delete_button = gr.Button("Delete my account", variant="stop")
                    with gr.Column(visible=False) as delete_confirm:
                        html('<div class="sb-note">This removes your account and every month of data. It cannot be undone.</div>')
                        with gr.Row():
                            confirm_delete_button = gr.Button("Delete for good", variant="stop")
                            keep_button = gr.Button("Keep my account")

    # ---------- Wiring ----------

    limit_boxes = [limit_overall, limit_necessities, limit_discretionary, limit_travel]
    session_outputs = [user_state, budget_state, month_state, open_state, auth_view, app_view]
    target_boxes = [target_needs, target_wants, target_savings]
    signed_in_outputs = session_outputs + [greeting, display_name, *limit_boxes, savings_goal, *target_boxes, login_password, new_password]

    for trigger in (login_button.click, login_password.submit):
        trigger(sign_in, inputs=[login_username, login_password], outputs=signed_in_outputs)
    for trigger in (create_button.click, new_password.submit):
        trigger(create_account, inputs=[new_display_name, new_username, new_password], outputs=signed_in_outputs)

    month_state.change(lambda key: f'<div class="sb-title">{budget.month_label(key)}</div>', inputs=month_state, outputs=month_title)
    demo.load(lambda key: f'<div class="sb-title">{budget.month_label(key)}</div>', inputs=month_state, outputs=month_title)
    previous_month.click(lambda key: (budget.shift_month(key, -1), None), inputs=month_state, outputs=[month_state, open_state])
    next_month.click(lambda key: (budget.shift_month(key, 1), None), inputs=month_state, outputs=[month_state, open_state])

    save_limits_button.click(save_settings, inputs=[user_state, budget_state, *limit_boxes, savings_goal], outputs=budget_state)
    save_targets_button.click(save_targets, inputs=[user_state, budget_state, *target_boxes], outputs=[budget_state, *target_boxes])
    use_rule_button.click(use_50_30_20, inputs=[user_state, budget_state], outputs=[budget_state, *target_boxes])
    save_name_button.click(save_display_name, inputs=[user_state, display_name], outputs=[greeting, display_name])
    change_password_button.click(change_password, inputs=[user_state, current_password, changed_password], outputs=[current_password, changed_password])

    sign_out_button.click(sign_out, outputs=session_outputs + [greeting, login_password])
    delete_button.click(lambda: gr.update(visible=True), outputs=delete_confirm)
    keep_button.click(lambda: gr.update(visible=False), outputs=delete_confirm)
    confirm_delete_button.click(delete_account, inputs=user_state, outputs=session_outputs + [greeting, login_password, delete_confirm])


if __name__ == "__main__":
    demo.launch(theme=design.build_theme(), css=design.CSS, share="--share" in sys.argv)  # Colab needs the public link


## `test_budget.py`

Automated tests for budget.py and storage.py.

In [ ]:
%%writefile test_budget.py
"""Tests for budget.py and storage.py. Run with: python -m pytest"""

from datetime import date

import pytest

import budget
import storage


# ---------- budget.py: input validation ----------

@pytest.mark.parametrize("typed, cents", [
    ("12", 1200), ("12.5", 1250), ("12.50", 1250), ("$1,200", 120000), (" 3 ", 300), (".99", 99), (7, 700),
])
def test_parse_amount_accepts_normal_input(typed, cents):
    assert budget.parse_amount(typed) == cents


@pytest.mark.parametrize("typed, message", [
    ("", "Enter an amount"), (None, "Enter an amount"), ("abc", "must be a number"), ("1.2.3", "must be a number"),
    ("-5", "negative"), ("0", "more than"), ("0.00", "more than"), ("1.234", "2 decimal"),
    ("10000001", "too large"), (".", "must be a number"),
])
def test_parse_amount_rejects_bad_input(typed, message):
    with pytest.raises(ValueError, match=message):
        budget.parse_amount(typed)


def test_optional_amount_treats_blank_and_zero_as_unset():
    assert budget.parse_optional_amount("") == 0
    assert budget.parse_optional_amount("0.00") == 0
    assert budget.parse_optional_amount("1,800") == 180000


def test_clean_name_trims_and_rejects_blank_or_long():
    assert budget.clean_name("  Rent   and  water ") == "Rent and water"
    with pytest.raises(ValueError, match="Enter a name"):
        budget.clean_name("   ")
    with pytest.raises(ValueError, match="40 characters"):
        budget.clean_name("x" * 41)


@pytest.mark.parametrize("cents, shown", [(120000, "$1,200"), (1250, "$12.50"), (-8500, "-$85"), (0, "$0"), (5, "$0.05")])
def test_format_money(cents, shown):
    assert budget.format_money(cents) == shown


# ---------- budget.py: calculations ----------

def make_budget():
    user_budget = budget.empty_budget()
    for section, name, amount in [
        ("income", "Salary", "3000"), ("income", "Freelance", "200"),
        ("necessities", "Rent", "1200"), ("necessities", "Groceries", "380.50"),
        ("discretionary", "Movies", "40"), ("travel", "Flights", "300"),
    ]:
        budget.add_item(user_budget, "2026-10", section, name, amount)
    return user_budget


def test_totals_and_leftover():
    month = budget.get_month(make_budget(), "2026-10")
    assert budget.section_total(month, "income") == 320000
    assert budget.total_expenses(month) == 120000 + 38050 + 4000 + 30000
    assert budget.leftover(month) == 320000 - 192050


def test_leftover_goes_negative_when_overspent():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-10", "income", "Job", "100")
    budget.add_item(user_budget, "2026-10", "necessities", "Rent", "185")
    assert budget.leftover(budget.get_month(user_budget, "2026-10")) == -8500


def test_empty_month_has_zero_everything():
    month = budget.get_month(budget.empty_budget(), "2030-01")
    assert budget.leftover(month) == 0 and budget.total_expenses(month) == 0


def test_remove_item():
    user_budget = make_budget()
    budget.remove_item(user_budget, "2026-10", "income", 1)
    assert budget.section_total(budget.get_month(user_budget, "2026-10"), "income") == 300000
    with pytest.raises(IndexError):
        budget.remove_item(user_budget, "2026-10", "income", 5)


def test_add_item_rejects_unknown_section_and_bad_amount_without_changing_anything():
    user_budget = budget.empty_budget()
    with pytest.raises(ValueError):
        budget.add_item(user_budget, "2026-10", "investments", "Stocks", "10")
    with pytest.raises(ValueError):
        budget.add_item(user_budget, "2026-10", "income", "Job", "lots")
    assert user_budget["months"] == {}


@pytest.mark.parametrize("spent, limit, state, label", [
    (100, 0, "none", ""),
    (50000, 100000, "good", "$500 left"),
    (90000, 100000, "warn", "Near limit, $100 left"),
    (100000, 100000, "warn", "Near limit, $0 left"),
    (104500, 100000, "over", "Over by $45"),
])
def test_meter_states(spent, limit, state, label):
    result = budget.meter(spent, limit)
    assert result["state"] == state and result["label"] == label
    assert 0 <= result["fraction"] <= 1


def test_savings_progress_ignores_overspent_months():
    user_budget = make_budget()
    budget.add_item(user_budget, "2026-11", "income", "Job", "100")
    budget.add_item(user_budget, "2026-11", "travel", "Trip", "500")
    assert budget.savings_progress(user_budget) == 320000 - 192050


@pytest.mark.parametrize("key, steps, result", [("2026-10", 1, "2026-11"), ("2026-12", 1, "2027-01"), ("2026-01", -1, "2025-12")])
def test_shift_month(key, steps, result):
    assert budget.shift_month(key, steps) == result


def test_month_key_and_label():
    assert budget.month_key(date(2026, 3, 9)) == "2026-03"
    assert budget.month_label("2026-10") == "October 2026"


# ---------- budget.py: editing a line ----------

def test_update_item_replaces_name_and_amount():
    user_budget = make_budget()
    budget.update_item(user_budget, "2026-10", "necessities", 0, "Rent and water", "1,250")
    month = budget.get_month(user_budget, "2026-10")
    assert month["necessities"][0] == {"name": "Rent and water", "amount": 125000}
    assert budget.section_total(month, "necessities") == 125000 + 38050


def test_update_item_rejects_bad_input_without_changing_anything():
    user_budget = make_budget()
    with pytest.raises(ValueError):
        budget.update_item(user_budget, "2026-10", "necessities", 0, "Rent", "-5")
    with pytest.raises(IndexError):
        budget.update_item(user_budget, "2026-10", "necessities", 9, "Rent", "5")
    assert budget.get_month(user_budget, "2026-10")["necessities"][0] == {"name": "Rent", "amount": 120000}


# ---------- budget.py: percentage targets ----------

@pytest.mark.parametrize("typed, targets", [
    (("50", "30", "20"), {"needs": 50, "wants": 30, "savings": 20}),
    (("60%", " 20 ", "20"), {"needs": 60, "wants": 20, "savings": 20}),
    (("", "", ""), {}),
    ((None, None, None), {}),
])
def test_parse_targets_accepts(typed, targets):
    assert budget.parse_targets(*typed) == targets


@pytest.mark.parametrize("typed, message", [
    (("50", "30", "10"), r"add up to 100% \(now 90%\)"),
    (("50", "", "20"), "Wants must be a whole number"),
    (("50.5", "29.5", "20"), "Needs must be a whole number"),
    (("150", "-30", "-20"), "Needs must be a whole number"),
])
def test_parse_targets_rejects(typed, message):
    with pytest.raises(ValueError, match=message):
        budget.parse_targets(*typed)


def test_target_meters_compare_spending_to_share_of_income():
    # Income $3,200; needs $1,580.50; wants $340; left $1,279.50.
    meters = budget.target_meters(budget.RULE_50_30_20, budget.get_month(make_budget(), "2026-10"))
    assert meters["needs"]["actual_percent"] == 49 and meters["needs"]["state"] == "warn"
    assert meters["needs"]["label"] == "Near limit, $19.50 left"
    assert meters["wants"]["actual_percent"] == 11 and meters["wants"]["state"] == "good"
    assert meters["savings"]["actual_percent"] == 40 and meters["savings"]["label"] == "On target"


def test_target_meters_flag_overspending_and_short_savings():
    user_budget = budget.empty_budget()
    budget.add_item(user_budget, "2026-10", "income", "Job", "1000")
    budget.add_item(user_budget, "2026-10", "necessities", "Rent", "600")
    budget.add_item(user_budget, "2026-10", "travel", "Trip", "300")
    meters = budget.target_meters(budget.RULE_50_30_20, budget.get_month(user_budget, "2026-10"))
    assert meters["needs"]["label"] == "Over by $100"
    assert meters["wants"]["state"] == "warn"
    assert meters["savings"]["state"] == "warn" and meters["savings"]["label"] == "$100 short"


def test_target_meters_empty_without_targets_or_income():
    month = budget.get_month(make_budget(), "2026-10")
    assert budget.target_meters({}, month) == {}
    assert budget.target_meters(budget.RULE_50_30_20, budget.empty_month()) == {}


# ---------- storage.py ----------

@pytest.fixture(autouse=True)
def temp_data_dir(tmp_path, monkeypatch):
    monkeypatch.setattr(storage, "DATA_DIR", str(tmp_path))


def test_create_account_and_sign_in():
    assert storage.create_account(" Mark_G ", "secret1", "Mark") == "mark_g"
    assert storage.sign_in("MARK_G", "secret1") == "mark_g"
    assert storage.get_display_name("mark_g") == "Mark"


@pytest.mark.parametrize("username, password, message", [
    ("ab", "secret1", "3-20"), ("has space", "secret1", "3-20"), ("../evil", "secret1", "3-20"), ("okname", "123", "at least 6"),
])
def test_create_account_rejects_bad_input(username, password, message):
    with pytest.raises(ValueError, match=message):
        storage.create_account(username, password)


def test_duplicate_username_rejected():
    storage.create_account("mark", "secret1")
    with pytest.raises(ValueError, match="taken"):
        storage.create_account("MARK", "secret2")


@pytest.mark.parametrize("username, password", [("mark", "wrong12"), ("nobody", "secret1"), ("", "")])
def test_sign_in_fails_with_one_message(username, password):
    storage.create_account("mark", "secret1")
    with pytest.raises(ValueError, match="Wrong username or password"):
        storage.sign_in(username, password)


def test_password_is_not_stored_in_plain_text(tmp_path):
    storage.create_account("mark", "secret1")
    assert "secret1" not in (tmp_path / "users.json").read_text()


def test_budget_round_trips_through_a_file():
    storage.create_account("mark", "secret1")
    user_budget = make_budget()
    user_budget["settings"]["savings_goal"] = 500000
    storage.save_budget("mark", user_budget)
    assert storage.load_budget("mark") == user_budget


def test_change_password():
    storage.create_account("mark", "secret1")
    with pytest.raises(ValueError, match="Wrong"):
        storage.change_password("mark", "nope123", "newpass1")
    storage.change_password("mark", "secret1", "newpass1")
    assert storage.sign_in("mark", "newpass1") == "mark"
    with pytest.raises(ValueError):
        storage.sign_in("mark", "secret1")


def test_delete_account_removes_user_and_budget(tmp_path):
    storage.create_account("mark", "secret1")
    storage.delete_account("mark")
    assert not (tmp_path / "mark.json").exists()
    with pytest.raises(ValueError):
        storage.sign_in("mark", "secret1")


## Run the tests

Every line should pass.

In [ ]:
!python -m pytest -q test_budget.py

## Run the app

In [ ]:
# Optional: keep accounts and budgets in Google Drive so they survive a runtime reset.
# Uncomment and run before the last cell.
# from google.colab import drive
# import os
# drive.mount("/content/drive")
# os.environ["SLICED_BREAD_DATA"] = "/content/drive/MyDrive/sliced-bread-data"


In [ ]:
# Start the app. Run this cell again after changing any file above.
import sys
import gradio as gr

gr.close_all()  # stop a copy started earlier in this session
for module in ("budget", "storage", "design", "app"):
    sys.modules.pop(module, None)  # re-import so edits to the cells above take effect

import app
import design

app.demo.launch(theme=design.build_theme(), css=design.CSS, share=True, height=900)